In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from dotenv import load_dotenv

load_dotenv()

emails = [
  {
    "subject": "周末见个面？",
    "content": """
        嗨 虎哥，
        我下周会去城里，不知道我们有没有机会一起喝杯咖啡？

        祝好，简
    """,
    "from": "jane@itcast.cn",
    "status": "unread"
  },
  {
    "subject": "周五会议",
    "content": """
        嗨 虎哥，
        非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？

        祝好，小李
    """,
    "from": "lixiaolong@itcast.cn",
    "status": "checked"
  }
]

@tool
def check_inbox() -> str:
  """查看邮箱中的邮件"""
  return str(emails)

@tool
def send_email(to: str, subject: str, body: str) -> str:
  """发送回复邮件"""
  return f"邮件已发送至{to}，主题：{subject}，内容：{body}"

human_in_loop_middleware = HumanInTheLoopMiddleware(
  interrupt_on={
    # 要人工确认的工具名字
    "send_email": {
      "description": "请确认邮件内容",
      "allowed_decisions": ["approve", "reject", "edit"]
    },
    # "check_inbox": {
    #   "description": "是否允许查看邮箱内容",
    #   "allowed_decisions": ["approve", "reject", "edit"]
    # }
  }
)
agent = create_agent(
  model="deepseek-flash",
  tools=[check_inbox, send_email],
  middleware=[human_in_loop_middleware],
  checkpointer=InMemorySaver(),
  system_prompt="你是一个邮件助手"
)

config = {
  "configurable": {"thread_id": "thread_1"}
}

In [12]:
response = agent.invoke(
  {"messages": [HumanMessage("帮我查看一下邮件")]},
  config
)
for message in response["messages"]:
  message.pretty_print()

================================ Human Message =================================

帮我查看一下邮件
================================== Ai Message ==================================

我现在帮您查看邮箱里的邮件。
Tool Calls:
  check_inbox (call_00_ET_fXe0MXO0tcDWwRWjfExE5167)
 Call ID: call_00_ET_fXe0MXO0tcDWwRWjfExE5167
  Args:
================================= Tool Message =================================
Name: check_inbox

[{'subject': '周末见个面？', 'content': '\n        嗨 虎哥，\n        我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n        祝好，简\n    ', 'from': 'jane@itcast.cn', 'status': 'unread'}, {'subject': '周五会议', 'content': '\n        嗨 虎哥，\n        非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n        祝好，小李\n    ', 'from': 'lixiaolong@itcast.cn', 'status': 'checked'}]
================================== Ai Message ==================================

您好，您的邮箱里目前有以下 **2 封邮件**：

---

**1. 周末见个面？**（未读 🔵）
- **发件人**：jane@itcast.cn
- **内容摘要**：简说她下周会去城里，想约您一起喝杯咖啡。

---

**2. 周五会议**（已读）
- **发件人**：lixiaolong@itcast.cn
- **内容摘要**：小李表示非常抱歉，他无法准时参加

In [13]:
response = agent.invoke(
  {"messages": [HumanMessage("帮我回复一下小李，看看下周什么时候有空，再约一个时间")]},
  config
)
print(response)
# for message in response["messages"]:
#   message.pretty_print()

{'messages': [HumanMessage(content='帮我查看一下邮件', additional_kwargs={}, response_metadata={}, id='2a947edc-0403-4113-b0f1-2cddf9733ebc'), AIMessage(content='我现在帮您查看邮箱里的邮件。', additional_kwargs={'refusal': None, 'reasoning_content': ''}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 349, 'total_tokens': 379, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 221}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '7570e7b8-19f7-4fb6-86cb-ec41f4053021', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1e8b6-46f1-7f22-8960-2c3a989869da-0', tool_calls=[{'name': 'check_inbox', 'args': {}, 'id': 'call_00_ET_fXe0MXO0tcDWwRWjfExE5167', 'type': 'tool_cal

In [14]:
from langgraph.types import Command

response = agent.invoke(
  Command(
    resume={
      "decisions": [
        {
          "type": "reject",
          "message": "语气需要正式一点"
        }
      ]
    }
  ),
  config
)

for message in response["messages"]:
  message.pretty_print()


================================ Human Message =================================

帮我查看一下邮件
================================== Ai Message ==================================

我现在帮您查看邮箱里的邮件。
Tool Calls:
  check_inbox (call_00_ET_fXe0MXO0tcDWwRWjfExE5167)
 Call ID: call_00_ET_fXe0MXO0tcDWwRWjfExE5167
  Args:
================================= Tool Message =================================
Name: check_inbox

[{'subject': '周末见个面？', 'content': '\n        嗨 虎哥，\n        我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n        祝好，简\n    ', 'from': 'jane@itcast.cn', 'status': 'unread'}, {'subject': '周五会议', 'content': '\n        嗨 虎哥，\n        非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n        祝好，小李\n    ', 'from': 'lixiaolong@itcast.cn', 'status': 'checked'}]
================================== Ai Message ==================================

您好，您的邮箱里目前有以下 **2 封邮件**：

---

**1. 周末见个面？**（未读 🔵）
- **发件人**：jane@itcast.cn
- **内容摘要**：简说她下周会去城里，想约您一起喝杯咖啡。

---

**2. 周五会议**（已读）
- **发件人**：lixiaolong@itcast.cn
- **内容摘要**：小李表示非常抱歉，他无法准时参加

In [15]:
response = agent.invoke(
  Command(
    resume={
      "decisions": [{"type": "approve"}]
    }
  ),
  config
)

for message in response["messages"]:
  message.pretty_print()

================================ Human Message =================================

帮我查看一下邮件
================================== Ai Message ==================================

我现在帮您查看邮箱里的邮件。
Tool Calls:
  check_inbox (call_00_ET_fXe0MXO0tcDWwRWjfExE5167)
 Call ID: call_00_ET_fXe0MXO0tcDWwRWjfExE5167
  Args:
================================= Tool Message =================================
Name: check_inbox

[{'subject': '周末见个面？', 'content': '\n        嗨 虎哥，\n        我下周会去城里，不知道我们有没有机会一起喝杯咖啡？\n\n        祝好，简\n    ', 'from': 'jane@itcast.cn', 'status': 'unread'}, {'subject': '周五会议', 'content': '\n        嗨 虎哥，\n        非常抱歉，我周五的会议无法准时参加了，能不能重新安排个时间？\n\n        祝好，小李\n    ', 'from': 'lixiaolong@itcast.cn', 'status': 'checked'}]
================================== Ai Message ==================================

您好，您的邮箱里目前有以下 **2 封邮件**：

---

**1. 周末见个面？**（未读 🔵）
- **发件人**：jane@itcast.cn
- **内容摘要**：简说她下周会去城里，想约您一起喝杯咖啡。

---

**2. 周五会议**（已读）
- **发件人**：lixiaolong@itcast.cn
- **内容摘要**：小李表示非常抱歉，他无法准时参加